### 데이터 받아오기

API로부터 데이터 받아오기

In [2]:
import requests
import json
import time
import os
from dotenv import load_dotenv

load_dotenv() 

# ── 상수 설정 ──────────────────────────────────────────
API_KEY    = os.getenv("DRUG_API_KEY")
# BASE_URL   = "	https://apis.data.go.kr/1471000/DrugSafeLetterService02"
ENDPOINT   = "https://apis.data.go.kr/1471000/DrugSafeLetterService02"
PAGE_SIZE  = 100   # 한 번 호출에 가져올 건수 (최대 100)
OUTPUT_DIR = "data"
OUTPUT_FILE = f"{OUTPUT_DIR}/safety_letters.json"
# ────────────────────────────────────────────────────────

다운파일로 시도

In [ ]:
# explore_data.py
import pandas as pd

try:
    df = pd.read_csv("의약품정보-병용금기.csv", encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv("의약품정보-병용금기.csv", encoding="cp949")

# ── 상세정보에서 자주 나오는 키워드 확인 ──────────────────
# 어떤 약물 카테고리가 많은지 파악하는 용도
keywords = ["항암", "독성", "금기", "부작용", "억제", "증가"]
for kw in keywords:
    count = df["상세정보"].str.contains(kw, na=False).sum()
    print(f"'{kw}' 포함 행 수: {count:,}")

print()

# ── 항암제 관련 성분 필터링 후 건수 확인 ──────────────────
# 항암제는 상세정보에 "항암", "독성", "종양" 등의 키워드가 많이 등장
CANCER_KEYWORDS = ["항암", "독성 증가", "세포독성", "종양", "백혈구"]
TARGET_KEYWORDS = [
    "항암",          # 108건 → 항암 직접 언급
    "독성 증가",     # 좁은 범위로 고위험만
    "세포독성",      # 항암제 특유 키워드
    "투여 금기",     # 명확한 금기 표현
]
mask = df["상세정보"].str.contains("|".join(CANCER_KEYWORDS), na=False)
df_cancer = df[mask]
print(f"항암 관련 병용금기: {len(df_cancer):,}건")

# ── 성분명A 기준 상위 20개 확인 ──────────────────────────
# 어떤 성분이 가장 많이 등장하는지 파악
print("\n성분명A 상위 20개:")
print(df_cancer["성분명A"].value_counts().head(20))

C:\Users\USER\AppData\Local\Temp\ipykernel_9260\71912245.py:5: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("의약품정보-병용금기.csv", encoding="utf-8")


'항암' 포함 행 수: 108
'독성' 포함 행 수: 37,257
'금기' 포함 행 수: 37,708
'부작용' 포함 행 수: 1
'억제' 포함 행 수: 9,438
'증가' 포함 행 수: 206,700

항암 관련 병용금기: 3,468건

성분명A 상위 20개:
성분명A
methotrexate                           2024
tacrolimus hydrate (as tacrolimus)      465
clarithromycin                          376
venetoclax                              246
itraconazole                            160
capecitabine                             72
minocycline hydrochloride                42
5-fluorouracil                           36
tetracycline hydrochloride               14
bupropion hydrochloride                  11
ritonavir                                 4
darunavir ethanolate (as darunavir)       4
posaconazole(micronized)                  4
voriconazole                              4
lithium carbonate                         2
lopinavir                                 2
ketoconazole                              2
Name: count, dtype: int64


In [32]:
# clean_dur.py
import pandas as pd
import json
import os
import hashlib

CSV_FILE    = "의약품정보-병용금기.csv"
OUTPUT_FILE = "dur_clean.json"
LIMIT       = 9500

TARGET_KEYWORDS = [
    "항암",
    "독성 증가",
    "세포독성",
    "투여 금기",
]

# 한글 컬럼명 → 영문 필드명 매핑 테이블
# 한눈에 보이도록 상수로 분리
FIELD_MAP = {
    "성분명A": "ingredientA",
    "성분명B": "ingredientB",
    "성분코드A": "ingredientCodeA",
    "성분코드B": "ingredientCodeB",
    "제품명A": "productA",
    "제품명B": "productB",
    "제품코드A": "productCodeA",
    "제품코드B": "productCodeB",
    "업체명A": "manufacturerA",
    "업체명B": "manufacturerB",
    "급여여부A": "coverageA",
    "급여여부B": "coverageB",
    "상세정보": "description",
    "고시번호": "noticeNo",
    "고시일자": "noticeDate",
    "비고": "remark",
}


def load(filepath: str) -> pd.DataFrame:
    """
    CSV 로드
    - low_memory=False: 비고 컬럼 혼재 타입 경고 억제
    - utf-8 실패 시 cp949 재시도
    """
    try:
        df = pd.read_csv(filepath, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="cp949", low_memory=False)

    print(f"로드 완료: {len(df):,}건 / 컬럼: {list(df.columns)}")
    return df


def filter_high_risk(df: pd.DataFrame) -> pd.DataFrame:
    """
    고위험 병용금기 조합만 추출
    - 상세정보 컬럼 기준 TARGET_KEYWORDS OR 조건 필터
    - na=False: NaN 행은 False 처리
    """
    pattern  = "|".join(TARGET_KEYWORDS)
    mask     = df["상세정보"].str.contains(pattern, na=False)
    filtered = df[mask].copy()

    print(f"\n필터링 결과: {len(filtered):,}건 (전체 {len(df):,}건 중)")
    for kw in TARGET_KEYWORDS:
        cnt = df["상세정보"].str.contains(kw, na=False).sum()
        print(f"  '{kw}': {cnt:,}건")

    return filtered


def build_documents(df: pd.DataFrame) -> list[dict]:
    """
    Azure AI Search 인덱싱용 문서 구조로 변환

    id:
    - 성분코드A + 성분코드B + 제품코드A + 제품코드B 조합 MD5 해시
    - Azure AI Search id는 영숫자/하이픈만 허용 → 해시로 안전하게 생성

    content:
    - 성분명 + 제품명 + 상세정보를 하나의 텍스트로 합침
    - 벡터 검색 + 키워드 검색 모두 이 필드를 대상으로 함
    - "A약이랑 B약 같이 먹어도 돼?" 질문에 매칭되는 핵심 필드
    """
    documents = []

    for _, row in df.iterrows():

        # 고유 ID: 성분코드+제품코드 조합 해시
        raw_id = (
            f"{row['성분코드A']}_"
            f"{row['성분코드B']}_"
            f"{row['제품코드A']}_"
            f"{row['제품코드B']}"
        )
        doc_id = hashlib.md5(raw_id.encode()).hexdigest()

        # 검색 핵심 텍스트
        content = (
            f"[병용금기] "
            f"{row['성분명A']} ({row['제품명A']}) + "
            f"{row['성분명B']} ({row['제품명B']}): "
            f"{row['상세정보']}"
        )
        content = content[:8000]
        doc = {
            "id":              doc_id,
            "content":         content,
            "ingredientA":     str(row["성분명A"]),
            "ingredientB":     str(row["성분명B"]),
            "ingredientCodeA": str(row["성분코드A"]),
            "ingredientCodeB": str(row["성분코드B"]),
            "productA":        str(row["제품명A"]),
            "productB":        str(row["제품명B"]),
            "productCodeA":    str(row["제품코드A"]),
            "productCodeB":    str(row["제품코드B"]),
            "manufacturerA":   str(row["업체명A"]),
            "manufacturerB":   str(row["업체명B"]),
            "coverageA":       str(row["급여여부A"]),
            "coverageB":       str(row["급여여부B"]),
            "description":     str(row["상세정보"]),
            "noticeNo":        str(row["고시번호"]),
            "noticeDate":      str(row["고시일자"]),
            "remark":          "" if pd.isna(row["비고"]) else str(row["비고"]),
        }
        documents.append(doc)

    return documents


def check_limit(documents: list[dict]) -> list[dict]:
    """
    무료 티어 1만 건 제한 체크
    - 초과 시 noticeDate 최신순으로 잘라냄
    """
    if len(documents) <= LIMIT:
        print(f"\n건수 확인: {len(documents):,}건 → 제한 이내 ✅")
        return documents

    print(f"\n⚠️  {len(documents):,}건 → {LIMIT:,}건으로 축소 (최신순)")
    documents = sorted(
        documents,
        key=lambda x: x["noticeDate"],
        reverse=True
    )[:LIMIT]
    return documents


def preview(documents: list[dict], n: int = 3) -> None:
    """
    저장 전 샘플 확인 및 키워드별 통계
    """
    print(f"\n── 샘플 {n}건 미리보기 ──")
    for doc in documents[:n]:
        print(f"  id           : {doc['id']}")
        print(f"  ingredientA  : {doc['ingredientA']}")
        print(f"  ingredientB  : {doc['ingredientB']}")
        print(f"  noticeDate   : {doc['noticeDate']}")
        print(f"  content      : {doc['content'][:120]}...")
        print("  " + "─" * 50)

    from collections import Counter
    kw_counts = Counter()
    for doc in documents:
        for kw in TARGET_KEYWORDS:
            if kw in doc["description"]:
                kw_counts[kw] += 1

    print(f"\n── 키워드별 최종 분포 ──")
    for kw, cnt in kw_counts.most_common():
        print(f"  '{kw}': {cnt:,}건")


def save(documents: list[dict]) -> None:
    """
    JSON 저장
    - ensure_ascii=False: 한글 유니코드 이스케이프 방지
    - indent=2: 가독성 확보 (Blob 업로드 후 확인 편의)
    """
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)
    print(f"\n저장 완료: {OUTPUT_FILE} ({len(documents):,}건)")


if __name__ == "__main__":
    df        = load(CSV_FILE)
    df        = filter_high_risk(df)
    documents = build_documents(df)
    documents = check_limit(documents)
    preview(documents)
    save(documents)


로드 완료: 837,837건 / 컬럼: ['Unnamed: 0', '성분명A', '성분코드A', '제품코드A', '제품명A', '업체명A', '급여여부A', '성분명B', '성분코드B', '제품코드B', '제품명B', '업체명B', '급여여부B', '고시번호', '고시일자', '상세정보', '비고']

필터링 결과: 39,802건 (전체 837,837건 중)
  '항암': 108건
  '독성 증가': 3,058건
  '세포독성': 0건
  '투여 금기': 36,744건

⚠️  39,802건 → 9,500건으로 축소 (최신순)

── 샘플 3건 미리보기 ──
  id           : 3642a1348f87d6eb2bd458aa0bff6fdd
  ingredientA  : 5-fluorouracil
  ingredientB  : tegafur
  noticeDate   : 2020-12-28
  content      : [병용금기] 5-fluorouracil (중외5-에프유주(플루오로우라실)_(0.5g/10mL)) + tegafur (티에스원캡슐25_(1캡슐)): 해당 약물 병용 시 gimeracil에 의해 fluoropyrimid...
  ──────────────────────────────────────────────────
  id           : aa3ff1f9e69b18d2a2b0c8579d867600
  ingredientA  : 5-fluorouracil
  ingredientB  : oteracil potassium
  noticeDate   : 2020-12-28
  content      : [병용금기] 5-fluorouracil (중외5-에프유주(플루오로우라실)_(0.25g/5mL)) + oteracil potassium (테고캡슐20_(1캡슐)): 해당 약물 병용 시 gimeracil에 의해 fluo...
  ──────────────────────────────────────────────────
  id           

### 콘텐츠 임베딩 추가

In [ ]:
import json
import time
import os
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

openai_client = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key        = os.getenv("AZURE_OPENAI_API_KEY"),
    api_version    = "2024-10-21",
)

search_client = SearchClient(
    endpoint   = os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name = os.getenv("AZURE_SEARCH_INDEX"),
    credential = AzureKeyCredential(os.getenv("AZURE_SEARCH_API_KEY")),
)

EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE      = 100  # 임베딩 API 호출 배치 크기
UPLOAD_BATCH    = 1000  # AI Search 업로드 배치 크기

# ── 1. dur_clean.json 로드 ─────────────────────────────────
with open("dur_clean.json", encoding="utf-8") as f:
    documents = json.load(f)

print(f"총 {len(documents):,}건 로드 완료")

# ── 2. content 필드 임베딩 생성 ───────────────────────────
def get_embeddings(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(
        model = EMBEDDING_MODEL,
        input = texts,
    )
    return [item.embedding for item in response.data]

print("임베딩 생성 시작...")
for i in range(0, len(documents), BATCH_SIZE):
    batch     = documents[i : i + BATCH_SIZE]
    texts     = [doc["content"] for doc in batch]
    vectors   = get_embeddings(texts)

    for doc, vector in zip(batch, vectors):
        doc["content_vector"] = vector

    print(f"  {min(i + BATCH_SIZE, len(documents)):,} / {len(documents):,} 완료")
    time.sleep(0.5)  # API 속도 제한 방지

print("임베딩 생성 완료 ✅")

# ── 3. Azure AI Search에 업로드 ───────────────────────────
print("\n업로드 시작...")
for i in range(0, len(documents), UPLOAD_BATCH):
    batch  = documents[i : i + UPLOAD_BATCH]
    result = search_client.upload_documents(documents=batch)
    print(f"  {min(i + UPLOAD_BATCH, len(documents)):,} / {len(documents):,} 업로드 완료")

print("업로드 완료 ✅")

총 9,500건 로드 완료
임베딩 생성 시작...
  100 / 9,500 완료
  200 / 9,500 완료
  300 / 9,500 완료
  400 / 9,500 완료
  500 / 9,500 완료
  600 / 9,500 완료
  700 / 9,500 완료
  800 / 9,500 완료
  900 / 9,500 완료
  1,000 / 9,500 완료
  1,100 / 9,500 완료
  1,200 / 9,500 완료
  1,300 / 9,500 완료
  1,400 / 9,500 완료
  1,500 / 9,500 완료
  1,600 / 9,500 완료
  1,700 / 9,500 완료
  1,800 / 9,500 완료
  1,900 / 9,500 완료
  2,000 / 9,500 완료
  2,100 / 9,500 완료
  2,200 / 9,500 완료
  2,300 / 9,500 완료
  2,400 / 9,500 완료
  2,500 / 9,500 완료
  2,600 / 9,500 완료
  2,700 / 9,500 완료
  2,800 / 9,500 완료
  2,900 / 9,500 완료
  3,000 / 9,500 완료
  3,100 / 9,500 완료
  3,200 / 9,500 완료
  3,300 / 9,500 완료
  3,400 / 9,500 완료
  3,500 / 9,500 완료
  3,600 / 9,500 완료
  3,700 / 9,500 완료
  3,800 / 9,500 완료
  3,900 / 9,500 완료
  4,000 / 9,500 완료
  4,100 / 9,500 완료
  4,200 / 9,500 완료
  4,300 / 9,500 완료
  4,400 / 9,500 완료
  4,500 / 9,500 완료
  4,600 / 9,500 완료
  4,700 / 9,500 완료
  4,800 / 9,500 완료
  4,900 / 9,500 완료
  5,000 / 9,500 완료
  5,100 / 9,500 완료
  5,200 / 9,500 완료
  

### 콘텐츠 임베딩 업로드

In [ ]:
import json
import time
import os
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

openai_client = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key        = os.getenv("AZURE_OPENAI_API_KEY"),
    api_version    = "2024-10-21",
)

search_client = SearchClient(
    endpoint   = os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name = os.getenv("AZURE_SEARCH_INDEX"),
    credential = AzureKeyCredential(os.getenv("AZURE_SEARCH_API_KEY")),
)

EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE      = 100  # 임베딩 API 호출 배치 크기
UPLOAD_BATCH    = 1000  # AI Search 업로드 배치 크기

# ── 1. dur_clean.json 로드 ─────────────────────────────────
with open("dur_clean.json", encoding="utf-8") as f:
    documents = json.load(f)

print(f"총 {len(documents):,}건 로드 완료")

# ── 2. content 필드 임베딩 생성 ───────────────────────────
def get_embeddings(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(
        model = EMBEDDING_MODEL,
        input = texts,
    )
    return [item.embedding for item in response.data]

print("임베딩 생성 시작...")
for i in range(0, len(documents), BATCH_SIZE):
    batch     = documents[i : i + BATCH_SIZE]
    texts     = [doc["content"] for doc in batch]
    vectors   = get_embeddings(texts)

    for doc, vector in zip(batch, vectors):
        doc["content_vector"] = vector

    print(f"  {min(i + BATCH_SIZE, len(documents)):,} / {len(documents):,} 완료")
    time.sleep(0.5)  # API 속도 제한 방지

print("임베딩 생성 완료 ✅")

# ── 3. Azure AI Search에 업로드 ───────────────────────────
print("\n업로드 시작...")
for i in range(0, len(documents), UPLOAD_BATCH):
    batch  = documents[i : i + UPLOAD_BATCH]
    result = search_client.upload_documents(documents=batch)
    print(f"  {min(i + UPLOAD_BATCH, len(documents)):,} / {len(documents):,} 업로드 완료")

print("업로드 완료 ✅")

### 챗봇 구현 그라디오

In [7]:
# chatbot.py
import os
import gradio as gr
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

openai_client = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key        = os.getenv("AZURE_OPENAI_API_KEY"),
    api_version    = "2024-10-21",
)

search_client = SearchClient(
    endpoint   = os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name = os.getenv("AZURE_SEARCH_INDEX"),
    credential = AzureKeyCredential(os.getenv("AZURE_SEARCH_API_KEY")),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")

SELECT_FIELDS = [
    "ingredientA", "ingredientB",
    "productA",    "productB",
    "description", "noticeDate",
    "content"
]

def search_drug_info(query: str, top: int = 5) -> list[dict]:
    """
    Hybrid 검색: 키워드(BM25) + 벡터 + 시맨틱 재랭킹
    + 성분명 필드 직접 필터 매칭 (ko.microsoft 분석기 한계 보완)
    """
    results = []
    seen_ids = set()

    # ── 1. 성분명 필드 직접 검색 (영문 성분명 매칭 보완) ──────
    # ingredientA/B는 ko.microsoft 분석기라 영어 검색이 부정확할 수 있음
    # search_fields로 해당 필드만 지정해 정확도 향상
    direct = search_client.search(
        search_text  = query,
        search_fields= ["ingredientA", "ingredientB"],
        top          = top,
        select       = SELECT_FIELDS,
    )
    for r in direct:
        d = dict(r)
        key = (d.get("ingredientA"), d.get("ingredientB"))
        if key not in seen_ids:
            seen_ids.add(key)
            results.append(d)

    # ── 2. Hybrid 검색: content 전문 + 벡터 + 시맨틱 재랭킹 ──
    vector_query = VectorizableTextQuery(
        text                = query,
        k_nearest_neighbors = top,
        fields              = "content_vector",
    )
    hybrid = search_client.search(
        search_text                 = query,
        vector_queries              = [vector_query],
        query_type                  = "semantic",
        semantic_configuration_name = "drug-index-semantic",
        top                         = top,
        select                      = SELECT_FIELDS,
    )
    for r in hybrid:
        d = dict(r)
        key = (d.get("ingredientA"), d.get("ingredientB"))
        if key not in seen_ids:
            seen_ids.add(key)
            results.append(d)

    return results[:top * 2]  # 최대 10건


def build_context(docs: list[dict]) -> str:
    if not docs:
        return ""
    context_parts = []
    for i, doc in enumerate(docs, 1):
        part = (
            f"[문서 {i}]\n"
            f"성분A: {doc.get('ingredientA', '')}\n"
            f"성분B: {doc.get('ingredientB', '')}\n"
            f"제품A: {doc.get('productA', '')}\n"
            f"제품B: {doc.get('productB', '')}\n"
            f"고시일자: {doc.get('noticeDate', '')}\n"
            f"상세정보: {doc.get('description', '')}\n"
        )
        context_parts.append(part)
    return "\n".join(context_parts)


def build_prompt(query: str, context: str) -> list[dict]:
    system_msg = """당신은 의약품 병용금기 전문 안내 챗봇입니다.
반드시 아래 규칙을 따르세요:

1. 제공된 문서 내용만을 근거로 답변하세요.
2. 답변 마지막에 반드시 출처(고시일자)를 명시하세요.
3. 문서에 없는 내용은 "해당 정보를 찾을 수 없습니다"라고 답하세요.
4. 의학적 최종 판단은 반드시 의사/약사와 상담하도록 안내하세요.
5. 한국어로 답변하세요."""

    user_msg = f"""아래 병용금기 문서를 참고해서 질문에 답변해주세요.

=== 참고 문서 ===
{context if context else "관련 문서를 찾을 수 없습니다."}

=== 질문 ===
{query}"""

    return [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]


def chat(query: str, history: list) -> str:
    if not query.strip():
        return "질문을 입력해주세요."
    docs     = search_drug_info(query)
    context  = build_context(docs)
    messages = build_prompt(query, context)
    response = openai_client.chat.completions.create(
        model       = DEPLOYMENT,
        messages    = messages,
        temperature = 0,
        max_tokens  = 1000,
    )
    return response.choices[0].message.content


with gr.Blocks(title="의약품 병용금기 챗봇") as demo:

    gr.Markdown("""
    # 💊 의약품 병용금기 안내 챗봇
    **식약처 DUR 데이터 기반** | Azure AI Search + GPT-4o-mini
    
    > ⚠️ 본 챗봇은 참고용이며 최종 판단은 의사/약사와 상담하세요.
    """)

    chatbot = gr.Chatbot(label="대화창", height=450)

    with gr.Row():
        query_box = gr.Textbox(
            placeholder="예: irinotecan 병용금기 약이 뭐야?",
            label="질문 입력",
            scale=4,
        )
        send_btn = gr.Button("전송", variant="primary", scale=1)

    gr.Examples(
        examples=[
            ["methotrexate와 병용 금기인 약이 뭐야?"],
            ["irinotecan 병용금기 성분 알려줘"],
            ["항암제 복용 중에 주의해야 할 약 조합 알려줘"],
            ["tacrolimus 병용금기 성분 알려줘"],
            ["독성이 증가하는 약 조합이 있어?"],
        ],
        inputs=query_box,
    )

    def respond(query, history):
        answer = chat(query, history)
        history = history + [{"role": "user", "content": query}, {"role": "assistant", "content": answer}]
        return history, ""

    send_btn.click(fn=respond, inputs=[query_box, chatbot], outputs=[chatbot, query_box])
    query_box.submit(fn=respond, inputs=[query_box, chatbot], outputs=[chatbot, query_box])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://58d919a69d9bb8e170.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
